# Unit 06 - Randomization Unit (Exercise)

**Atoms:** `U06-A3`, `U06-A4`, `U06-A5` · **Runtime:** ~25 seconds

## Without code
Expect: session `SE` < user `SE` < cluster `SE` (session-level looks over-precise); contamination > 50 users; design effect at rho=0.1 is 2.9.

## 1. The question
Compute `SE` at three units, count contaminated users, compute design effect.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

In [ ]:
n_users = 500
n_clusters = 25
users_per_cluster = n_users // n_clusters
cluster_id = np.repeat(np.arange(n_clusters), users_per_cluster)
D_user = np.random.binomial(1, 0.5, n_clusters)[cluster_id]
cluster_effect = np.random.normal(0, 0.03, n_clusters)[cluster_id]
Y_user = np.random.binomial(1, np.clip(0.15 + 0.02*D_user + cluster_effect, 0, 1))
rows = []
for u in range(n_users):
    for s in range(np.random.poisson(2)):
        rows.append({'user_id': u, 'cluster_id': cluster_id[u], 'D_user': D_user[u], 'Y': np.random.binomial(1, Y_user[u])})
sessions = pd.DataFrame(rows)

## 4. TODO - standard errors
Compute `SE` at user and cluster level. Assert cluster `SE` > user `SE`.

In [ ]:
def ate_se(df, treat_col, outcome='Y'):
    g = df.groupby(treat_col)[outcome]
    m, v, n = g.mean(), g.var(), g.count()
    return (m[1]-m[0], np.sqrt(v[1]/n[1]+v[0]/n[0]))

user_df = sessions.groupby(['user_id','D_user'])['Y'].mean().reset_index()
cluster_df = sessions.groupby(['cluster_id','D_user'])['Y'].mean().reset_index()
# TODO
se_user = None
se_cluster = None
assert se_user is not None and se_cluster is not None
assert se_cluster > se_user
print('SE user:', round(se_user,4), 'SE cluster:', round(se_cluster,4))

## 5. TODO - contamination
Assign `D_session` ~ Bernoulli(0.5) per session. Count users with both arms.

In [ ]:
wrong = sessions.copy()
wrong['D_session'] = np.random.binomial(1, 0.5, len(wrong))
# TODO
n_contam = None
assert n_contam is not None
print('Contaminated users:', n_contam)
assert n_contam > 50

## 6. TODO - design effect
For `m=20`, `rho=0.1`, compute `1 + (m-1)*rho`. Expect **2.9**.

In [ ]:
# TODO
design_effect = None
assert design_effect is not None
print('Design effect:', design_effect)
assert abs(design_effect - 2.9) < 0.01

**Takeaway:** Match the unit to the treatment. **Unit:** [V1](../V1/units/unit-06-assignment-mechanism/README.md) · [V2](../V2/units/unit-06-assignment-mechanism/README.md)

## Hints
- Reuse `ate_se` from demo.
- `n_contam = (wrong.groupby('user_id')['D_session'].nunique()>1).sum()`.

## Spoiler

```python
_, se_user = ate_se(user_df, 'D_user')
_, se_cluster = ate_se(cluster_df, 'D_user')

n_contam = (wrong.groupby('user_id')['D_session'].nunique() > 1).sum()

design_effect = 1 + (20 - 1) * 0.1
```